# Laboruebung 2: Systemdynamik

In [ ]:
import control as ctrl
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import jet
from scipy.integrate import solve_ivp

## Aufgabe 2.5: Stabilitaet

In [ ]:
# Teilaufgabe 1: Untersuchung auf Stabilität

# Definition der Übertragungsfunktionen in Form der Polynomkoeffizienten.
# Wir nutzen eine Liste, und jedes Listenelement besteht aus einem
# Dictionary. Die Keys des Dictionaries sind 'zaehler' und 'nenner'.
uebertragungs_funktionen = ( \
                {'zaehler': [1],     'nenner': [1, 0]},      \
                {'zaehler': [1],     'nenner': [1, 0, 0]},   \
                {'zaehler': [1, 1],  'nenner': [1, 3, 2]},   \
                {'zaehler': [1, -2], 'nenner': [1, -1, -2]}, \
                {'zaehler': [1],     'nenner': [1, 0, 9]},   \
                {'zaehler': [1, 3],  'nenner': [1, 2, 4]},   \
               )

# Wir nutzen eine Schleife, um über alle Übertragungsfunktionen zu
# iterieren. Die Funktion 'enumerate' gibt uns dabei auch den Laufindex.
for idx, ue_funktion in enumerate(uebertragungs_funktionen):
    # Erstellung der Uebertragungsfunktion
    G = ctrl.tf(ue_funktion['zaehler'], ue_funktion['nenner'])
    # Berechnung der Pole der Uebertragungsfunktion
    poles = ctrl.poles(G)
    # Ausgabe der Pole
    print(f'Die Pole der Uebertragungsfunktion G_{idx+1}(s)=\n{G}\nsind')
    print(poles)
    # Alle Pole in der offenen linken Halbebene --> stabil.
    # Ein Pole in der rechten Halbebene --> instabil.
    # Einfache Pole auf der imaginaeren Achse --> grenzstabil.
    # Mehrfache Pole auf der imaginaeren Aches --> instabil.
    if all(np.real(poles) < 0.0):
        stabilitaet = 'stabil'
    elif any(np.real(poles) > 0.0):
        stabilitaet = 'instabil'
    else:
        _, vielfachheit = np.unique(poles, return_counts=True)
        # Hinweis: Der Underscore _ ignoriert den Rueckgabewert (in dem Fall
        # das Array mit eindeutigen Polen).
        # Wir sind nur an der Vielfachheit der Pole interessiert, und
        # nutzen deshalb 'return_counts=True'.
        if vielfachheit[0] > 1:
            stabilitaet = 'instabil'
        else:
            stabilitaet = 'grenzstabil'
    print(f'\nDas System G_{idx+1}(s) ist {stabilitaet}.\n\n')
    print('----------------------------------------------------')

In [ ]:
# Teilaufgabe 2: Einfluss der Rückführung auf die Stabilität

G1 = ctrl.tf([1], [1,1,1])
# Pole ohne Rueckfuehrung
poles_without_feedback = ctrl.poles(G1)
print(f'Pole der Strecke (ohne Rueckfuehrung): {poles_without_feedback}')
if all(np.real(poles_without_feedback) < 0.0):
    print('Die Strecke G1 ist stabil.')

# Mit Rueckfuehrung: Variante 1
G2 = ctrl.tf([5] ,[1, 1])
G_feedback = ctrl.feedback(G1, G2)
poles_with_feedback_variant_1 = ctrl.poles(G_feedback)
print(f'Pole des Gesamtsystems mit Rueckfuehrung (Variante 1): {poles_with_feedback_variant_1}')
# --> Das System ist instabil.

# Mit Rueckfuehrung: Variante 2
G2 = ctrl.tf([5] ,[5, 1])
G_feedback = ctrl.feedback(G1, G2)
poles_with_feedback_variant_2 = ctrl.poles(G_feedback)
print(f'Pole des Gesamtsystems mit Rueckfuehrung (Variante 2): {poles_with_feedback_variant_2}')
# --> Das System ist stabil.

# Wir sehen, dass wir durch eine Rückführung (feedback) die
# Stabilität verändern können.
# Dadurch können wir instabile Strecken stabilisieren (dies ist eine
# der Kernaufgaben der Regelungstechnik).
# Durch die Rückführung können aber auch bereits von sich aus
# stabile Strecken destabilisiert werden,
# wie wir hier in dieser Teilaufgabe gesehen haben.
# Dies kann sogar dann passieren, wenn wir (wie hier) eine
# negative Rückführung vorliegen haben.

## Aufgabe 2.6: Pol-Nullstelle-Diagramme

In [ ]:
# Definition der Übertragungsfunktionen in Form der Polynomkoeffizienten.
# Wir nutzen eine Liste, und jedes Listenelement besteht aus einem
# Dictionary. Die Keys des Dictionaries sind 'zaehler' und 'nenner'.
uebertragungs_funktionen = ( \
                {'zaehler': [1], 'nenner': [1, 0]},       \
                {'zaehler': [1], 'nenner': [1, 0, 0]},    \
                {'zaehler': [1], 'nenner': [2, 1]},       \
                {'zaehler': [1], 'nenner': [2, -1]},      \
                {'zaehler': [1], 'nenner': [8, 1]},       \
                {'zaehler': [1], 'nenner': [8, -1]},      \
                {'zaehler': [1], 'nenner': [1, 0, 1]},    \
                {'zaehler': [1], 'nenner': [1, 0, 9]},    \
                {'zaehler': [1], 'nenner': [1, 0.2, 1]},  \
                {'zaehler': [1], 'nenner': [1, -0.2, 1]}, \
               )

### Teilaufgabe 1

In [ ]:
# Teilaufgabe 1: Plotten der Pol-Nullstellen-Diagramme
# Dabei überprüfen wir auch gleich auf Stabilität.
# Hinweis: Es gibt hier nur Pole, keine Nullstellen.

for idx, ue_funktion in enumerate(uebertragungs_funktionen):
    G = ctrl.tf(ue_funktion['zaehler'], ue_funktion['nenner'], name=f'G_{idx+1}(s)')
    plt.figure(idx+1)
    plt.clf()
    ctrl.pzmap(G)
    plt.grid(True)
    s1 = str(G)
    G_tf_string = '\n'.join(s1.split('\n')[-3:])
    plt.annotate(f'{G_tf_string}', xy=(0.6, 0.7), xycoords='figure fraction')
    plt.title(f'Pol-Nullstellendiagramm fuer G_{idx+1}(s)')
    # Ueberpruefung auf Stabilitaet.
    # Zuerst erfolgt die Berechnung der Pole.
    poles = ctrl.poles(G)
    print(f'Pole: {poles}\n')
    # Alle Pole in der offenen linken Halbebene --> stabil.
    # Ein Pole in der rechten Halbebene --> instabil.
    # Einfache Pole auf der imaginaeren Achse --> grenzstabil.
    # Mehrfache Pole auf der imaginaeren Aches --> instabil.
    if all(np.real(poles) < 0.0):
        stabilitaet = 'stabil'
    elif any(np.real(poles) > 0.0):
        stabilitaet = 'instabil'
    else:
        _, vielfachheit = np.unique(poles, return_counts=True)
        # Hinweis: Der Underscore _ ignoriert den Rueckgabewert (in dem Fall
        # das Array mit eindeutigen Polen).
        # Wir sind nur an der Vielfachheit der Pole interessiert, und
        # nutzen deshalb 'return_counts=True'.
        if vielfachheit[0] > 1:
            stabilitaet = 'instabil'
        else:
            stabilitaet = 'grenzstabil'
    print(f'Das System G_{idx+1}(s)={G} ist {stabilitaet}\n\n')


### Teilaufgabe 2

In [ ]:
# Teilaufgabe 2: Darstellung der Impulsantworten mitsamt der Pol-Nullstellen-Diagramme
    
t_eval = np.linspace(0,20,1000)

for idx, ue_funktion in enumerate(uebertragungs_funktionen):
    G = ctrl.tf(ue_funktion['zaehler'], ue_funktion['nenner'])
    poles = ctrl.poles(G)
    fig = plt.figure(idx+1, figsize=(8,3))
    plt.clf()
    # Pol-Nullstellen-Diagramm (nur Pole hier)
    ax = fig.add_subplot(1, 2, 1)
    ax.scatter(np.real(poles), np.imag(poles), marker='x')
    ax.grid(True)
    ax.set(xlabel='Re(s)')
    ax.set(ylabel='Im(s)')
    ax.set(xlim=(-4,4))
    ax.set(ylim=(-4,4))
    # Impulsantwort
    g = ctrl.impulse_response(G, t_eval)
    ax = fig.add_subplot(1, 2, 2)
    ax.plot(g[0], g[1])
    ax.grid(True)
    ax.set(xlabel='t')
    ax.set(ylabel=f'g_{idx+1}(t)')
    ax.set(ylim=(-3,3))
    fig.suptitle(f'G_{idx+1}(s)')

# Die Lage des Realteils der Polstellen beeinflusst die Auf- oder Abklingzeit.
# Je weiter links ein stabiler Pol sich befindet, desto schneller
# klingt die Impulsantwort ab.
# Je weiter rechts sich ein instabiler Pol befindet, desto schneller
# divergiert die Impulsantwort.

# Die Lage des Imaginärteils der Polstellen beeinflusst
# die Schwingungsfrequenz.
# Die Polstellen treten immer als konjugiert komplexe Pole auf.
# Je weiter die Polstellen von der reellen Achse entfernt sind
# (je grösser also der Betrag des Imaginärteils ist),
# desto höher (also schneller) ist die Schwingungsfrequenz.



### Teilaufgabe 3

In [ ]:
# Teilaufgabe 3: Darstellung der Sprungantworten mitsamt der Pol-Nullstellen-Diagramme
    
t_eval = np.linspace(0,20,1000)

for idx, ue_funktion in enumerate(uebertragungs_funktionen):
    G = ctrl.tf(ue_funktion['zaehler'], ue_funktion['nenner'])
    poles = ctrl.poles(G)
    fig = plt.figure(idx+1, figsize=(8,3))
    plt.clf()
    # Pol-Nullstellen-Diagramm (nur Pole hier)
    ax = fig.add_subplot(1, 2, 1)
    ax.scatter(np.real(poles), np.imag(poles), marker='x')
    ax.grid(True)
    ax.set(xlabel='Re(s)')
    ax.set(ylabel='Im(s)')
    ax.set(xlim=(-4,4))
    ax.set(ylim=(-4,4))
    # Sprungantwort
    h = ctrl.step_response(G, t_eval)
    ax = fig.add_subplot(1, 2, 2)
    ax.plot(h[0], h[1])
    ax.grid(True)
    ax.set(xlabel='t')
    ax.set(ylabel=f'h_{idx+1}(t)')
    ax.set(ylim=(-3,3))
    fig.suptitle(f'G_{idx+1}(s)')

# Die Lage des Realteils der Polstellen beeinflusst die Einschwingzeit
# (bei stabilen Systemen) bzw. die Geschwindigkeit, in der das
# Signal divergiert (bei instabilen Systemen).
# Je weiter links ein stabiler Pol sich befindet, desto schneller
# schwingt sich die Sprungantwort ein.
# Je weiter rechts sich ein instabiler Pol befindet, desto schneller
# divergiert die Sprungantwort.

# Die Lage des Imaginärteils der Polstellen beeinflusst
# die Schwingungsfrequenz.
# Die Polstellen treten immer als konjugiert komplexe Pole auf.
# Je weiter die Polstellen von der reellen Achse entfernt sind
# (je grösser also der Betrag des Imaginärteils ist),
# desto höher (also schneller) ist die Schwingungsfrequenz.


## Aufgabe 2.7: Zeitbereichsverhalten eines PT2 Systems

Definition der Uebertragungsfunktion

In [ ]:
params = { 'omega_n': 1.0,
           'zeta': 0.3 }

H = ctrl.tf([params['omega_n']**2], [1.0, 2*params['zeta']*params['omega_n'], params['omega_n']**2])

### Berechnung der Impulsantwort

In [ ]:
# Berechnung via impulse_response
t_eval = np.linspace(0,10,200)
t, imp_response = ctrl.impulse_response(H, t_eval)

# Vergleich mit inverser Laplace Transformation
params['sigma'] = params['zeta']*params['omega_n']
params['omega_d'] = params['omega_n'] * np.sqrt(1.0 - params['zeta']**2)
imp_response_analytical = params['omega_n']/np.sqrt(1.0-params['zeta']**2) * \
                            np.exp(-params['sigma']*t_eval) * \
                            np.sin(params['omega_d'] * t_eval)

fig = plt.figure('Impulsantwort')
plt.clf()
plt.plot(t*params['omega_n'], imp_response, label='Impulsantwort via control package')
plt.plot(t_eval*params['omega_n'], imp_response_analytical, linestyle='--', label='Analytische Lsg. Impulsantwort')
plt.grid()
plt.xlabel('t*omega_n')
plt.legend()
plt.show()


### Berechnung der Sprungantwort

In [ ]:
t, step_response = ctrl.step_response(H, t_eval)

fig = plt.figure('Sprungantwort')
plt.clf()
plt.plot(t*params['omega_n'], step_response)
plt.grid()
plt.xlabel('t*omega_n')
plt.show()

### Kurven fuer verschiedene Werte von zeta

In [ ]:
zeta_range = np.arange(0.0, 1.1, 0.1)

fig = plt.figure('Impuls- und Sprungantworten fuer verschiedene zeta', figsize=(9,4))
plt.clf()

ax = fig.add_subplot(1,2,1)
for idx, zeta in enumerate(zeta_range):
    H = ctrl.tf([params['omega_n']**2], [1.0, 2*zeta*params['omega_n'], params['omega_n']**2])
    t, imp_response = ctrl.impulse_response(H, t_eval)
    ax.plot(t*params['omega_n']/np.pi, imp_response, label=f'zeta = {zeta:2.1f}', color=jet(zeta))

ax.set(xlabel='t*omega_n/pi')
ax.grid()
ax.legend()
ax.set(title='Impulsantworten')

ax = fig.add_subplot(1,2,2)
for idx, zeta in enumerate(zeta_range):
    H = ctrl.tf([params['omega_n']**2], [1.0, 2*zeta*params['omega_n'], params['omega_n']**2])
    t, step_response = ctrl.step_response(H, t_eval)
    ax.plot(t*params['omega_n']/np.pi, step_response, label=f'zeta = {zeta:2.1f}', color=jet(zeta))

ax.set(xlabel='t*omega_n/pi')
ax.grid()
ax.legend()
ax.set(title='Sprungantworten')

plt.show()

## Aufgabe 2.8: Uebertragungsverhalten

Definition der Strecke
$$G(s)=\frac{1}{s(s+1)}=\frac{1}{s^2+s}$$
und des Reglers
$$K(s)=10 \frac{s+2}{s+10}$$

In [ ]:
# Plant (Strecke)
s = ctrl.tf('s')
G = ctrl.tf([1], [1, 1, 0], inputs='u', outputs='y')

# Controller (Regler)
K = ctrl.tf([10, 20], [1, 10], inputs='e', outputs='u')

### Gesamtsystem

In [ ]:
sum_junction = ctrl.summing_junction(inputs=['w', '-y'], outputs='e')
closed_loop = ctrl.interconnect([G, K, sum_junction], inputs='w', outputs='y', name='Gesamtsystem')

print(closed_loop)
closed_loop_tf = ctrl.tf(closed_loop)
print(closed_loop_tf)

### Sprungantworten

Ueberpruefung im Zeitbereich anhand von Sprungantworten

In [ ]:
# Sprungantwort des ungeregelten Systems
t, y = ctrl.step_response(G)

plt.figure('Plant step response')
plt.clf()
plt.plot(t, y)
plt.grid(True)
plt.xlabel('t'), plt.ylabel('h')
plt.show()

# Sprungantwort des geregelten Systems
t, y = ctrl.step_response(closed_loop_tf)

plt.figure('Closed loop step response')
plt.clf()
plt.plot(t, y)
plt.grid(True)
plt.xlabel('t'), plt.ylabel('h')
plt.show()


# Simulation mit scipy
def ode(t, x):
    # Rechte Seite u
    # Hier 1, da wir die Sprungantwort suchen
    rhs = 1.0
    return closed_loop.dynamics(t, x, rhs)

x_0 = np.zeros((3,))
sol = solve_ivp(ode, [0.0, 7.0], x_0)

t_scipy = sol.t
y_scipy = np.dot(closed_loop.C, sol.y).flatten()

# Plot the solution
plt.figure('Closed loop step response via integration with scipy')
plt.clf()
plt.plot(t_scipy, y_scipy, label='Solution via scipy')
plt.plot(t, y, label='Solution via control toolbox', linestyle='--')
plt.grid(True)
plt.xlabel('t'), plt.ylabel('h')
plt.legend()
plt.show()